
# DDRI 군집별 2차 비교 취합 노트북

- 목적: 1차 결과와 2차 결과를 비교해 `ddri_cluster_second_round_comparison_summary.csv`를 재생성한다.
- 입력:
  - `summary_aggregation/output/data/ddri_cluster_model_metrics_collection_template.csv`
  - `works/05_prediction_long/output/team_cluster_second_round_outputs/*.csv`
- 출력: `summary_aggregation/output/data/ddri_cluster_second_round_comparison_summary.csv`


In [1]:

from pathlib import Path
import pandas as pd

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
AGG_DIR = ROOT / 'works/05_prediction_long/cheng80/summary_aggregation'
OUTPUT_DATA_DIR = AGG_DIR / 'output/data'
FIRST_PATH = OUTPUT_DATA_DIR / 'ddri_cluster_model_metrics_collection_template.csv'
SECOND_DIR = ROOT / 'works/05_prediction_long/output/team_cluster_second_round_outputs'

first_df = pd.read_csv(FIRST_PATH, encoding='utf-8-sig')


## 1. 1차 최종 test 결과와 2차 최종 test 결과 비교

In [2]:

second_rows = []
for csv_path in sorted(SECOND_DIR.glob('ddri_*_second_round_model_metrics.csv')):
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    station_group = df['station_group'].iloc[0]
    test_row = df[df['split'] == 'test_2025_refit'].sort_values('rmse').iloc[0]
    second_rows.append({
        'station_group': station_group,
        'second_round_best_model': test_row['model'],
        'second_round_test_rmse': round(float(test_row['rmse']), 4),
        'second_round_test_mae': round(float(test_row['mae']), 4),
        'second_round_test_r2': round(float(test_row['r2']), 4),
    })
second_df = pd.DataFrame(second_rows)

first_best = (
    first_df[first_df['best_model_flag'] == 1][['cluster_code', 'station_group', 'model', 'rmse', 'mae', 'r2']]
    .rename(columns={
        'model': 'first_round_best_model',
        'rmse': 'first_round_test_rmse',
        'mae': 'first_round_test_mae',
        'r2': 'first_round_test_r2',
    })
)

comp = first_best.merge(second_df, on='station_group', how='inner')
comp['rmse_delta'] = (comp['second_round_test_rmse'] - comp['first_round_test_rmse']).round(4)
comp['mae_delta'] = (comp['second_round_test_mae'] - comp['first_round_test_mae']).round(4)
comp['r2_delta'] = (comp['second_round_test_r2'] - comp['first_round_test_r2']).round(4)
comp = comp[['cluster_code','station_group','first_round_best_model','first_round_test_rmse','first_round_test_mae','first_round_test_r2','second_round_best_model','second_round_test_rmse','second_round_test_mae','second_round_test_r2','rmse_delta','mae_delta','r2_delta']].sort_values('cluster_code')

out_path = OUTPUT_DATA_DIR / 'ddri_cluster_second_round_comparison_summary.csv'
comp.to_csv(out_path, index=False, encoding='utf-8-sig')
print(comp.to_string(index=False))
print() 
print('saved:', out_path)


cluster_code station_group first_round_best_model  first_round_test_rmse  first_round_test_mae  first_round_test_r2 second_round_best_model  second_round_test_rmse  second_round_test_mae  second_round_test_r2  rmse_delta  mae_delta  r2_delta
   cluster00     업무/상업 혼합형          LightGBM_RMSE                 0.8113                0.5439               0.3212           LightGBM_RMSE                  0.8085                 0.5403                0.3259     -0.0028    -0.0036    0.0047
   cluster01  아침 도착 업무 집중형          LightGBM_RMSE                 1.3462                0.8042               0.6398           LightGBM_RMSE                  1.3324                 0.7868                0.6471     -0.0138    -0.0174    0.0073
   cluster02        주거 도착형          LightGBM_RMSE                 0.8088                0.5059               0.4987           LightGBM_RMSE                  0.8059                 0.5053                0.5022     -0.0029    -0.0006    0.0035
   cluster03       생활권 혼합형      